# COMP7103 Lab 8 · Mine rules with Apriori, then check an LLM summary

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab8/Lab8_rules_check.ipynb)

You work for an online gift shop in the UK (home decorations, bags, kitchen and party items, Christmas goods). You have its invoices from June to November 2011. In Part A you turn the invoices into baskets and mine association rules with Apriori (Lecture 7). In Part B an LLM has summarised 200 mined rules for the sales team and cites ten of them by rule ID; you **recompute every cited rule** on the data and report how many citations are wrong.

| Time | What you do |
|---|---|
| 5 min | Brief |
| 12 min | Part A: baskets and Apriori |
| 18 min | Part B: verify 10 cited rules |
| 10 min | Share-out |

This lab is ungraded practice for the final exam. Data: Online Retail II (UCI Machine Learning Repository, id 502), CC BY 4.0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup (brief, 5 min)

This cell loads the invoice lines: one row per invoice and product. To keep the file small, it holds only the 100 products that appear on the most invoices; `data/SOURCE.txt` says how the file was made.

In [ ]:
import os, time, urllib.request
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)
pd.set_option("display.max_colwidth", None)   # show long product names in full

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab8/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 8 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 8 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab8")
    return os.path.join("data", name)

lines = pd.read_csv(get("invoices_uk_2011.csv"), dtype=str)
print(f"{len(lines):,} invoice lines, {lines['InvoiceNo'].nunique():,} invoices, {lines['Description'].nunique()} products")
lines.head()

## 1 · Part A: invoices into baskets (4 min)

A **basket** (a transaction, Lecture 7) is the set of products on one invoice. The one-hot table `baskets` has one row per invoice and one column per product; a cell is `True` when the invoice contains that product. Fill in the two column names: the invoice number gives the rows, the product description gives the columns.

In [ ]:
baskets = pd.crosstab(lines[___], lines[___]) > 0      # rows: invoices, columns: products
print("baskets × products:", baskets.shape)
items_per_basket = baskets.sum(axis=1)      # True counts as 1, so the sum of a row = its number of products
print("products per basket: median", items_per_basket.median(), "· mean", round(items_per_basket.mean(), 1))
baskets.iloc[:5, :4]                         # the first 5 baskets and the first 4 products

The **support** of an itemset $X$ is the share of the $N$ baskets that contain all of $X$: $s(X) = |\{t : X \subseteq t\}| \,/\, N$. For one product it is the mean of its column. Notation: Lecture 7 writes $\text{sup}(X)$ for the *count* of baskets that contain $X$, so $s(X) = \text{sup}(X)/N$; below, $c(X \rightarrow Y)$ is the lecture's $P(Y \mid X)$, and $X \rightarrow Y$ is the lecture's $X \Rightarrow Y$. The 15 most frequent products:

In [ ]:
item_support = baskets.mean()               # the mean of a True/False column = its share of True = s({item})
item_support = item_support.sort_values()   # smallest first, so the 15 largest are at the end
ax = item_support.tail(15).plot.barh(color="#6d4fc2", figsize=(7, 4.5))
ax.set_xlabel("support (share of baskets)")
ax.set_ylabel("")
ax.set_title("The 15 most frequent products")
plt.tight_layout()
plt.show()

## 2 · Part A: Apriori at the suggested thresholds (5 min)

`support()` (given) computes $s(X)$; `apriori()` (given) is the level-wise algorithm of Lecture 7: start from the frequent single products; join two frequent $k$-itemsets that share their first $k-1$ items into a candidate; **prune** the candidate if one of its $k$-subsets is not frequent (the Apriori property); count the rest and keep those with support ≥ `min_support`. It returns a dictionary {itemset: support}.

In [ ]:
def support(columns, itemset):
    """s(X): the share of baskets that contain every item of the itemset X.
    columns[item] is an array with one True/False per basket: does the basket contain the item?"""
    has_all = columns[itemset[0]]
    for item in itemset[1:]:
        has_all = has_all & columns[item]              # True only if the basket also has this item
    return has_all.mean()                              # the share of True values

def apriori(baskets, min_support):
    """Level-wise Apriori (Lecture 7). baskets: one-hot table (rows = baskets, columns = items, True/False).
    Returns a dictionary {frequent itemset (a sorted tuple of items): its support}."""
    columns = {}
    for item in baskets.columns:
        columns[item] = baskets[item].to_numpy()       # one True/False per basket

    frequent = {}

    # Level 1: the frequent single items
    level = []
    for item in baskets.columns:                       # the columns are sorted A to Z
        s = support(columns, (item,))
        if s >= min_support:
            frequent[(item,)] = s
            level.append((item,))

    # Level k + 1 from level k, until no itemset is frequent
    k = 1
    while len(level) > 0:
        next_level = []
        for i in range(len(level)):
            for j in range(i + 1, len(level)):
                a = level[i]
                b = level[j]
                # Join: two k-itemsets that share their first k - 1 items
                if a[:-1] != b[:-1]:
                    continue
                candidate = a + (b[-1],)
                # Prune: every k-subset of the candidate must be frequent (the Apriori property)
                all_subsets_frequent = True
                for subset in combinations(candidate, k):
                    if subset not in frequent:
                        all_subsets_frequent = False
                if not all_subsets_frequent:
                    continue
                # Count: keep the candidate if it is frequent
                s = support(columns, candidate)
                if s >= min_support:
                    frequent[candidate] = s
                    next_level.append(candidate)
        level = next_level
        k = k + 1
    return frequent

`make_rules()` splits every frequent itemset $X \cup Y$ into rules $X \rightarrow Y$. Fill in the two formulas:

$$c(X \rightarrow Y) = \frac{s(X \cup Y)}{s(X)}, \qquad \text{lift}(X \rightarrow Y) = \frac{c(X \rightarrow Y)}{s(Y)}.$$

In the code, `support` is $s(X \cup Y)$ and `freq[X]` is $s(X)$. Lift > 1: buying $X$ makes $Y$ more likely than average; lift < 1: less likely (a negative association).

In [ ]:
def make_rules(freq, min_confidence):
    """All rules X -> Y from the frequent itemsets, with confidence >= min_confidence."""
    rows = []
    for itemset in freq:
        support = freq[itemset]                         # s(X ∪ Y)
        for size in range(1, len(itemset)):             # the number of items on the left side X
            for X in combinations(itemset, size):
                Y = tuple([item for item in itemset if item not in X])    # the other items
                confidence = ___ / freq[X]
                if confidence >= min_confidence:
                    lift = confidence / ___
                    rows.append([X, Y, support, confidence, lift])
    rules = pd.DataFrame(rows, columns=["antecedent", "consequent", "support", "confidence", "lift"])
    # Highest lift first; equal lift: higher support first
    rules = rules.sort_values(["lift", "support"], ascending=False, ignore_index=True)
    return rules

Run Apriori at the suggested thresholds: **min support 2%** (`0.02`) and **min confidence 30%** (`0.3`). The rules come sorted by lift.

In [ ]:
t = time.time()
freq = apriori(baskets, min_support=___)
rules = make_rules(freq, min_confidence=___)
print(f"{len(freq)} frequent itemsets, {len(rules)} rules ({time.time() - t:.1f} s)")

sizes = {}                                  # itemset size -> how many frequent itemsets have that size
for itemset in freq:
    size = len(itemset)
    sizes[size] = sizes.get(size, 0) + 1
print("itemsets by size:", sizes)

In [ ]:
def as_set(items):
    """('A', 'B') -> '{A, B}'"""
    return "{" + ", ".join(items) + "}"

def show(rules, n=10):
    """The first n rules, readable: support and confidence in %, lift with 2 decimals."""
    rows = []
    for i in rules.index[:n]:
        rule = rules.loc[i]
        rows.append({"rule": as_set(rule["antecedent"]) + " → " + as_set(rule["consequent"]),
                     "support": f"{rule['support']:.2%}",          # 0.0310 -> 3.10%
                     "confidence": f"{rule['confidence']:.1%}",    # 0.452 -> 45.2%
                     "lift": f"{rule['lift']:.2f}"})
    return pd.DataFrame(rows, index=rules.index[:n])

show(rules, 10)

## 3 · Part A: your own threshold (3 min)

Choose one other pair of thresholds, for example a lower support or a higher confidence, and compare the counts and the top rules. Keep min support at 1% (`0.01`) or above: below that the number of rules explodes (at 0.75% and confidence 5% there are about 120,000 rules).

In [ ]:
MY_SUPPORT = ___
MY_CONFIDENCE = ___
my_freq = apriori(baskets, MY_SUPPORT)
my_rules = make_rules(my_freq, MY_CONFIDENCE)
print(f"min support {MY_SUPPORT:.1%}, min confidence {MY_CONFIDENCE:.0%}: {len(my_freq)} frequent itemsets, {len(my_rules)} rules")
show(my_rules, 10)

Do the top rules make sense for a gift shop? Look for products sold in several colours or designs of one range (teacups, lunch boxes, Christmas decorations). Lift favours rare itemsets: a lower support threshold brings in rules with very high lift that hold for only about 80 of the 7,900 baskets (1%).

## 4 · Part B: the mined rule file and the LLM summary (3 min)

So that everyone checks the same rules, Part B uses a fixed rule file, `rules_top200.csv`. It was mined with the same `apriori()` and `make_rules()` at min support 0.75% and min confidence 5% (about 120,000 rules). The file keeps the 192 rules with the highest support and the 8 rules with the lowest lift (all below 0.95). The rules are numbered **R001–R200 by lift**, highest first; these IDs belong to this file only (your tables in Part A have no IDs).

In [ ]:
file_rules = pd.read_csv(get("rules_top200.csv"))
# In the file a side is written "A, B"; split it into the tuple ("A", "B")
file_rules["antecedent"] = file_rules["antecedent"].str.split(", ").map(tuple)
file_rules["consequent"] = file_rules["consequent"].str.split(", ").map(tuple)
BY_ID = file_rules.set_index("id")            # look up a rule by its ID: BY_ID.loc["R012"]

def read_number(text, name):
    """The number written after a word: read_number("support 3.10%, lift 5.41", "lift") -> 5.41."""
    start = text.find(name)
    if start == -1:
        return np.nan                         # the word is not in the text
    digits = ""
    for ch in text[start + len(name):]:
        if ch.isdigit() or ch == ".":
            digits = digits + ch              # part of the number
        elif digits != "":
            break                             # the number has ended
    if digits.strip(".") == "":
        return np.nan
    return float(digits.strip("."))

def read_side(text):
    """'GREEN TEACUP, PINK TEACUP' -> ('GREEN TEACUP', 'PINK TEACUP')"""
    return tuple([item.strip() for item in text.split(",")])

def citations(text):
    """Find the citations [R012: {left} → {right}, support 3.10%, confidence 45.2%, lift 5.41] in a text."""
    rows = []
    for piece in text.split("[")[1:]:         # each piece starts right after a "["
        if "]" not in piece:
            continue
        inside = piece[:piece.index("]")]     # the text between "[" and "]"
        rule_id = inside.replace(":", " ").replace(",", " ").split(" ")[0]    # e.g. "R012"
        if not (rule_id.startswith("R") and rule_id[1:].isdigit()):
            continue                          # not a citation of a rule
        body = inside[len(rule_id):]
        # The rule {left} → {right}: the first two pairs of curly brackets
        left_start = body.find("{")
        left_end = body.find("}", left_start)
        right_start = body.find("{", left_end)
        right_end = body.find("}", right_start)
        if -1 in (left_start, left_end, right_start, right_end):
            antecedent = ()                   # only an ID, e.g. [R012]
            consequent = ()
        else:
            antecedent = read_side(body[left_start + 1:left_end])
            consequent = read_side(body[right_start + 1:right_end])
        rows.append({"id": rule_id, "antecedent": antecedent, "consequent": consequent,
                     "support %": read_number(body, "support"),
                     "confidence %": read_number(body, "confidence"),
                     "lift": read_number(body, "lift")})
    cited = pd.DataFrame(rows)
    cited.index = range(1, len(rows) + 1)     # number the citations 1, 2, 3, ...
    cited.index.name = "#"
    return cited

def same_rule(x1, y1, x2, y2):
    """True if the rules x1 → y1 and x2 → y2 have the same items on each side (in any order)."""
    return set(x1) == set(x2) and set(y1) == set(y2)

def find_in_file(x, y):
    """The ID of the rule x → y in rules_top200.csv, or None if the file does not have it."""
    for i in file_rules.index:
        if same_rule(x, y, file_rules.loc[i, "antecedent"], file_rules.loc[i, "consequent"]):
            return file_rules.loc[i, "id"]
    return None

def check(cited):
    """For each citation: the rule at that ID in the file, then the stated numbers next to a recount."""
    for i in cited.index:
        c = cited.loc[i]
        rule_id = c["id"]
        x = c["antecedent"]
        y = c["consequent"]
        written_out = len(x) > 0              # False when the citation gives only an ID, e.g. [R012]

        # Line 1: the rule as the text cites it
        if written_out:
            print(f"{i}. {rule_id}   cited:  {as_set(x)} → {as_set(y)}")
        else:
            print(f"{i}. {rule_id}   cited: (no rule written out)")

        # Line 2: the rule with this ID in the file
        if rule_id in BY_ID.index:
            f = BY_ID.loc[rule_id]
            if not written_out or same_rule(x, y, f["antecedent"], f["consequent"]):
                note = "(the same rule)"
            else:
                note = "(A DIFFERENT RULE)"
            print(f"         file:   {as_set(f['antecedent'])} → {as_set(f['consequent'])}   {note}")
        else:
            print(f"         file:   there is no {rule_id} in rules_top200.csv")
            if not written_out:
                print()
                continue                      # no rule to recount

        # Line 3 (only if needed): is the cited rule in the file under another ID?
        if written_out:
            found_id = find_in_file(x, y)
            if found_id is None:
                print("         the cited rule is nowhere in the file")
            elif found_id != rule_id:
                print(f"         the cited rule is in the file as {found_id}")
        else:
            x = f["antecedent"]               # no rule written out: recount the file's rule
            y = f["consequent"]

        # Lines 4 and 5: the stated numbers, then your recount
        r = recount(x, y)
        print(f"         stated:  support {c['support %']:.2f}%   confidence {c['confidence %']:.1f}%   lift {c['lift']:.2f}")
        print(f"         recount: support {r['support']:.2%}   confidence {r['confidence']:.1%}   lift {r['lift']:.2f}\n")

show(file_rules.set_index("id"), 200).iloc[[0, 1, 2, 100, 191, 192, 199]]

The 200 rules were given to an LLM with the message in `data/llm_prompt.txt`: summarise them for the sales team in 3 to 5 themes and cite exactly ten rules with their numbers. Here is the answer. It comes from Qwen2.5-1.5B-Instruct, a small open model, and was edited for this lab: we added the numbers to its citations and planted some errors, so your error rate measures how well you check, not this model. `data/SOURCE.txt` says how.

In [ ]:
summary = open(get("llm_summary.txt")).read()
print(summary)

## 5 · Part B: verify the ten cited rules (15 min)

`recount()` (given) counts a rule directly in your `baskets`, without Apriori and without the file:

In [ ]:
def recount(antecedent, consequent):
    """Count the rule X → Y directly in the baskets. Returns support, confidence and lift."""
    X = list(antecedent)
    Y = list(consequent)
    unknown = [item for item in X + Y if item not in baskets.columns]
    if len(unknown) > 0:
        print("not a product in the baskets:", unknown)
        return pd.Series({"support": np.nan, "confidence": np.nan, "lift": np.nan})
    has_x = baskets[X].all(axis=1)            # True for the baskets that contain every item of X
    has_y = baskets[Y].all(axis=1)            # ... every item of Y
    has_xy = has_x & has_y                    # ... every item of X and of Y
    support = has_xy.mean()                   # s(X ∪ Y)
    confidence = has_xy.sum() / has_x.sum()   # s(X ∪ Y) / s(X)
    lift = confidence / has_y.mean()          # c(X → Y) / s(Y)
    return pd.Series({"support": support, "confidence": confidence, "lift": lift}).round(4)

recount(["PINK REGENCY TEACUP AND SAUCER"], ["GREEN REGENCY TEACUP AND SAUCER"])   # R008 in the file

`citations()` finds the ten citations in the summary; `check()` prints, for each one, the rule at that ID in the file and the numbers the summary states next to your recount. Fill in the text to read.

In [ ]:
cited = citations(___)
print(len(cited), "citations")
check(cited)

Give each citation **one** verdict; if more than one fits, take the first in this list.

1. `"never mined"`: the ID is not in `rules_top200.csv`, or the rule the summary describes is not the rule at that ID (and it is nowhere in the file).
2. `"lift < 1 called interesting"`: the rule's lift is below 1, but the summary sells it as interesting, strong or a cross-selling opportunity. Read the sentence around the citation.
3. `"wrong number"`: a stated support, confidence or lift differs from your recount by more than rounding (support by more than 0.05 percentage points, confidence by more than 0.5 points, lift by more than 0.05).
4. `"correct"`: none of the above.

The **error rate** is the number of wrong citations (any verdict except correct) divided by the 10 cited rules.

In [ ]:
verdicts = [___]      # ten strings, in citation order: "correct", "wrong number", "lift < 1 called interesting" or "never mined"

assert len(verdicts) == 10, "give one verdict per citation"
assert set(verdicts) <= {"correct", "wrong number", "lift < 1 called interesting", "never mined"}, "check the spelling"
n_wrong = 0
for v in verdicts:
    if v != "correct":
        n_wrong = n_wrong + 1
error_rate = n_wrong / ___
print(pd.Series(verdicts, index=cited["id"]).to_string())
print(f"\nerror rate: {error_rate:.0%}")

In [ ]:
# One box per citation: green if correct, red otherwise ("\n" starts a new line in the box)
LABEL = {"correct": "correct", "wrong number": "wrong\nnumber",
         "lift < 1 called interesting": "lift < 1\ncalled\ninteresting", "never mined": "never\nmined"}
fig, ax = plt.subplots(figsize=(9, 1.6))
for i in range(len(verdicts)):
    v = verdicts[i]
    if v == "correct":
        colour = "#2e9d4f"   # green
    else:
        colour = "#d64545"   # red
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, color=colour))
    ax.text(i + 0.45, 0.5, f"{i + 1}\n{LABEL[v]}", ha="center", va="center", color="white", fontsize=8)
ax.set_xlim(0, 10)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title(f"The ten cited rules (green: correct, red: wrong) · error rate {error_rate:.0%}")
plt.show()

## 6 · Optional: audit your own LLM's summary

Try this after the lab, or if you finish early. Run the next cell; it prints the same message the cached summary came from (the 200 rules, no personal data). Paste it into any chat LLM, paste the whole answer between the triple quotes in the cell after, and check its citations the same way. If your LLM writes its citations in another form, look them up by hand with `BY_ID.loc["R012"]` and `recount([...], [...])`.

In [ ]:
print(open(get("llm_prompt.txt")).read())

In [ ]:
my_reply = """
paste the LLM's answer here
"""
# No LLM? The cached summary came from the same message:
# my_reply = open(get("llm_summary.txt")).read()
my_cited = citations(my_reply)
print(len(my_cited), "citations found")
check(my_cited)

## 7 · Share-out (10 min)

- Report your verdict for each of the ten cited rules (1 to 10, in the order they appear) and your **error rate**; we collect them on the class board.
- Which citation was hardest to judge, and why?
- Part A: how many frequent itemsets and rules did you get at 2% / 30%, and at your own thresholds? Name one top rule that makes sense for a gift shop.
- The habit from the lecture: an LLM can group and describe hundreds of rules, but every rule it cites is recomputed on the data.